In [1]:
!pip -q install -U transformers gradio

In [2]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "openai-community/gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading {MODEL_NAME} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-2 has no separate padding token, so use its EOS token for padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)
model.eval()

print("Model loaded successfully!")

Loading openai-community/gpt2 on cuda...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
STYLE_PREFIXES = {
    "Fantasy": "A fantasy story. ",
    "Mystery": "A mystery story. ",
    "Science Fiction": "A science fiction story. ",
    "Adventure": "An adventure story. ",
    "Horror": "A horror story. ",
    "Romance": "A romantic story. ",
    "Comedy": "A funny story. ",
    "Realistic Fiction": "A realistic fiction story. ",
}


def generate_story_text(opening, style, temperature, max_new_tokens=120):
    """Generate a continuation and return the story plus its tokenizer token count."""
    if not opening or not opening.strip():
        return "Please enter an opening sentence or story prompt.", 0

    opening = opening.strip()
    style_prefix = STYLE_PREFIXES.get(style, "A story. ")
    model_prompt = style_prefix + opening

    # Leave enough room in GPT-2's 1024-token context for the generated continuation.
    max_input_tokens = max(1, 1024 - int(max_new_tokens) - 5)
    encoded = tokenizer(
        model_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    )
    encoded = {key: value.to(device) for key, value in encoded.items()}
    input_length = encoded["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **encoded,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=max(0.1, float(temperature)),
            top_p=0.92,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    continuation = tokenizer.decode(
        output_ids[0][input_length:], skip_special_tokens=True
    ).strip()

    if continuation:
        # Avoid an extra space if the continuation begins with punctuation.
        separator = "" if continuation[0] in ",.!?;:" else " "
        story = opening + separator + continuation
    else:
        story = opening

    token_count = len(tokenizer.encode(story, add_special_tokens=False))
    return story, token_count


def generate_story(opening, style, temperature, max_new_tokens):
    story, token_count = generate_story_text(opening, style, temperature, max_new_tokens)
    return story, f"**Token count:** {token_count}"


def compare_temperatures(opening, style, max_new_tokens):
    """Generate comparable stories with the same prompt at low and high temperatures."""
    low_story, low_count = generate_story_text(opening, style, 0.5, max_new_tokens)
    high_story, high_count = generate_story_text(opening, style, 1.2, max_new_tokens)
    low_result = f"{low_story}\n\n---\n**Token count:** {low_count} | **Temperature:** 0.5"
    high_result = f"{high_story}\n\n---\n**Token count:** {high_count} | **Temperature:** 1.2"
    return low_result, high_result

In [ ]:
with gr.Blocks(title="GPT-2 Interactive Story Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 📖 GPT-2 Interactive Story Generator\n"
        "Enter an opening sentence, choose a style, and control how predictable or varied the continuation should be."
    )

    with gr.Row():
        with gr.Column(scale=2):
            opening_input = gr.Textbox(
                label="Opening sentence / prompt",
                placeholder="Example: At midnight, the old clock began to speak...",
                lines=3,
                value="At midnight, the old clock began to speak.",
            )
            style_input = gr.Dropdown(
                choices=list(STYLE_PREFIXES.keys()),
                value="Mystery",
                label="Writing style",
            )
            temperature_input = gr.Slider(
                minimum=0.2,
                maximum=1.5,
                value=0.8,
                step=0.1,
                label="Temperature (lower = more predictable; higher = more varied)",
            )
            length_input = gr.Slider(
                minimum=40,
                maximum=200,
                value=120,
                step=20,
                label="Maximum new tokens",
            )
            generate_button = gr.Button("Generate Story", variant="primary")
        with gr.Column(scale=3):
            story_output = gr.Textbox(label="Generated story", lines=14, interactive=False)
            token_output = gr.Markdown("**Token count:** —")

    generate_button.click(
        fn=generate_story,
        inputs=[opening_input, style_input, temperature_input, length_input],
        outputs=[story_output, token_output],
    )

    gr.Markdown("## Temperature comparison\nGenerate two versions from the same opening and style. Only the temperature changes.")
    compare_button = gr.Button("Compare Temperature 0.5 vs 1.2")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Low temperature (0.5)")
            low_output = gr.Markdown()
        with gr.Column():
            gr.Markdown("### High temperature (1.2)")
            high_output = gr.Markdown()
    compare_button.click(
        fn=compare_temperatures,
        inputs=[opening_input, style_input, length_input],
        outputs=[low_output, high_output],
    )

    gr.Markdown(
        "**Note:** GPT-2 is a pretrained base model, not a modern instruction-following model. "
        "Some stories may be repetitive or drift away from the selected style. Temperature comparison is stochastic, "
        "so each run may produce different text."
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_504/1550800782.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="GPT-2 Interactive Story Generator", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://551bd48aa15a78c708.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
